# NULL handling

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-03-null-handling.ipynb)

`NULL` means *unknown*. It is not zero, not an empty string, and not "no value" in the sense of a valid
answer. Because it is unknown, comparisons with `NULL` are unknown too, and a filter keeps only rows where
the condition is definitely `TRUE`. Most NULL bugs in analytics come from forgetting that rule.

Previous: [WHERE vs HAVING](sql-02-where-vs-having.ipynb). Next: [CASE expressions](sql-04-case-expressions.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## Three-valued logic

Every comparison yields `TRUE`, `FALSE` or `UNKNOWN`. `WHERE` keeps `TRUE` only. So `NULL = NULL` is not
`TRUE`; it is `UNKNOWN`. To ask "are these both missing?" use `IS NULL`, or the null-safe equality operator
`<=>`.

In [ ]:
check(
    """
    SELECT
        (NULL = NULL)   AS equals_null,
        (NULL <=> NULL) AS null_safe_equal,
        (1 = NULL)      AS one_equals_null,
        (NULL IS NULL)  AS is_null
    """,
    [(None, True, None, True)],
    "NULL comparisons",
)

## A small table with missing values

`contacts` has a missing email and a missing score. The next checks use it to show how filters, aggregates
and sorting treat those gaps.

In [ ]:
create_view(
    "contacts",
    "id INT, email STRING, score INT",
    [
        (1, "a@x.com", 10),
        (2, None, None),
        (3, "b@x.com", 30),
    ],
)

### Filters drop unknowns

`email <> 'a@x.com'` is `UNKNOWN` for row 2, so row 2 is dropped. If you want to keep it, say so
explicitly with `IS NULL`.

In [ ]:
check(
    "SELECT id FROM contacts WHERE email <> 'a@x.com' ORDER BY id",
    [(3,)],
    "not-equal drops the NULL email",
)
check(
    "SELECT id FROM contacts WHERE email IS NULL OR email <> 'a@x.com' ORDER BY id",
    [(2,), (3,)],
    "explicit NULL handling keeps it",
)

### Aggregates skip NULL

`COUNT(*)` counts rows. `COUNT(column)` counts non-NULL values. `SUM` and `AVG` ignore NULL inputs, so the
average score is over two rows, not three. That is usually what you want, but it is worth knowing when a
denominator changes silently.

In [ ]:
check(
    """
    SELECT
        COUNT(*)     AS all_rows,
        COUNT(email) AS with_email,
        COUNT(score) AS with_score,
        AVG(score)   AS avg_score,
        SUM(score)   AS total_score
    FROM contacts
    """,
    [(3, 2, 2, 20.0, 40)],
    "aggregates over NULLs",
)

### `COALESCE` and `NULLIF`

`COALESCE(a, b, ...)` returns the first non-NULL argument, which is how you supply a default. `NULLIF(a, b)`
returns `NULL` when the two are equal, which is how you turn a sentinel such as `0` into a real NULL before
averaging.

In [ ]:
check(
    "SELECT id, COALESCE(score, 0) AS score_filled FROM contacts ORDER BY id",
    [(1, 10), (2, 0), (3, 30)],
    "COALESCE supplies a default",
)
check(
    "SELECT NULLIF(0, 0) AS sentinel_removed, NULLIF(5, 0) AS kept",
    [(None, 5)],
    "NULLIF turns a sentinel into NULL",
)

### Sorting NULLs

Spark sorts `NULL` first in ascending order by default. Add `NULLS LAST` when you want them at the end, as
most reports do.

In [ ]:
check(
    "SELECT id FROM contacts ORDER BY score",
    [(2,), (1,), (3,)],
    "default ascending order puts NULL first",
)
check(
    "SELECT id FROM contacts ORDER BY score NULLS LAST",
    [(1,), (3,), (2,)],
    "NULLS LAST moves NULL to the end",
)

## The `NOT IN` trap

`NOT IN (list)` is true only when the value differs from *every* item. If the list contains a `NULL`, no
comparison can be proven `TRUE`, so the whole predicate is `UNKNOWN` and nothing is returned. `NOT EXISTS`
compares one row at a time, so the `NULL` simply matches nobody.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

In [ ]:
create_view("tiny_orders", "customer_id INT", [(1,), (None,)])

In [ ]:
check(
    "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)",
    [],
    "NOT IN with a NULL returns nothing",
)
check(
    """
    SELECT c.customer_id
    FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """,
    [(2,), (3,), (4,), (5,), (6,), (7,), (8,)],
    "NOT EXISTS ignores the NULL",
)

## Takeaways

- `= NULL` is never true. Use `IS NULL`, or `<=>` when you need null-safe equality.
- `COUNT(column)` and `COUNT(*)` answer different questions.
- Prefer `NOT EXISTS` to `NOT IN` when the subquery can contain NULL.
- Decide the NULL sort order explicitly with `NULLS FIRST` or `NULLS LAST`.